In [1]:
import pandas as pd

df = pd.read_csv("../data/Sample - Superstore.csv", encoding="latin1")
df["Order Date"] = pd.to_datetime(df["Order Date"])

snapshot = df["Order Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID").agg(
    Recency=("Order Date", lambda x: (snapshot - x.max()).days),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum"),
)
rfm.describe()

,Recency,Frequency,Monetary
count,793.000000,793.000000,793.000000
mean,147.802018,6.316520,2896.848500
std,186.211051,2.550885,2628.670117
min,1.000000,1.000000,4.833000
25%,31.000000,5.000000,1146.050000
50%,76.000000,6.000000,2256.394000
75%,184.000000,8.000000,3785.276000
max,1166.000000,17.000000,25043.050000


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

X = StandardScaler().fit_transform(np.log1p(rfm))

for k in range(2, 7):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    print(k, round(silhouette_score(X, labels), 3))

2 0.35
3 0.255
4 0.255
5 0.251
6 0.243


In [3]:
rfm["Cluster"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X)
rfm.groupby("Cluster").agg(
    Customers=("Monetary", "size"),
    Recency=("Recency", "mean"),
    Frequency=("Frequency", "mean"),
    Monetary=("Monetary", "mean"),
).round(1)

,Customers,Recency,Frequency,Monetary
Cluster,,,,
0,199,19.7,6.8,2712.9
1,253,235.2,4.9,1936.8
2,258,103.3,8.6,4773.0
3,83,326.6,2.6,432.2
